In [ ]:
import sys
import pandas as pd
import geopandas as gpd
import os
import glob
from datetime import datetime
from parcel_calculations import add_improvement_ratio_fields

sys.path.append("..")

from cloud_utils import get_feature_data_with_geometry

pd.set_option("display.max_columns", None)

# Parameters for Cincinnati parcels (Hamilton County, OH)
dataset_name = "CAGIS_Open_Data"
base_url = "https://services.arcgis.com/JyZag7oO4NteHGiq/ArcGIS/rest/services"
layer_id = 12  # Hamilton_County_Parcel_Polygons

SCRAPE_DATA = 0  # set to 0 or 1 as required
DATA_DIR = os.path.join("data", "cincinnati")
os.makedirs(DATA_DIR, exist_ok=True)

if SCRAPE_DATA == 1:
    parcel_gdf = get_feature_data_with_geometry(dataset_name, base_url, layer_id)
    today_str = datetime.now().strftime("%Y_%m_%d")
    out_path = os.path.join(DATA_DIR, f"cincinnati_parcels_{today_str}.parquet")
    parcel_gdf.to_parquet(out_path, index=False)
    print(f"✅ Saved new scrape to {out_path}")

else:
    files = glob.glob(os.path.join(DATA_DIR, "cincinnati_parcels_*.parquet"))
    if not files:
        raise FileNotFoundError(f"No parcel files found in {DATA_DIR}. Set SCRAPE_DATA=1 to scrape.")

    files_sorted = sorted(
        files,
        key=lambda x: datetime.strptime(
            os.path.basename(x).replace("cincinnati_parcels_", "").replace(".parquet", ""),
            "%Y_%m_%d"
        ),
        reverse=True
    )
    latest_file = files_sorted[0]
    print(f"✅ Loading most recent scrape: {latest_file}")
    parcel_gdf = gpd.read_parquet(latest_file)

print(f"✅ Loaded as {type(parcel_gdf).__name__} | CRS = {getattr(parcel_gdf, 'crs', None)}")


In [ ]:
# Display the head of the dataframe with all columns shown using display.max_columns option
with pd.option_context('display.max_columns', None):
    display(parcel_gdf.head())


In [ ]:
# Create a 'parcel_link' column based on AUDPCLID column to match:
# https://wedge.hcauditor.org/view/re/<AUDPCLID>/2025/summary

if "AUDPCLID" in parcel_gdf.columns:
    parcel_gdf["parcel_link"] = (
        "https://wedge.hcauditor.org/view/re/"
        + parcel_gdf["AUDPCLID"].astype(str)
        + "/2025/summary"
    )
else:
    raise KeyError("AUDPCLID column not found in parcel_gdf.")

# Backfill 'link' for UI compatibility
if "link" not in parcel_gdf.columns:
    parcel_gdf["link"] = parcel_gdf.get("parcel_link")


In [ ]:
# Display all columns for the row(s) where AUDPCLID is '0014100040029'
with pd.option_context('display.max_columns', None):
    # Show all with AUDPCLID == "14100040029"
    display(parcel_gdf[parcel_gdf["AUDPCLID"] == "14100040029"])
    # Also show .head() for BOOK == 141, PAGE == '0004', PARCEL == '0029'
    mask = (
        (parcel_gdf["BOOK"].astype(str) == "141") &
        (parcel_gdf["PAGE"].astype(str).str.zfill(4) == "0004") &
        (parcel_gdf["PARCEL"].astype(str).str.zfill(4) == "0029")
    )
    display(parcel_gdf[mask].head())

    mask = (
        (parcel_gdf["BOOK"].astype(str) == "141") &
        (parcel_gdf["PAGE"].astype(str).str.zfill(4) == "0004") &
        (parcel_gdf["PARCEL"].astype(str).str.zfill(4) == "0037")
    )
    display(parcel_gdf[mask].head())


In [ ]:
n_dupes_all = parcel_gdf.duplicated(subset=["BOOK", "PAGE", "PARCEL"]).sum()
print(f"Number of duplicate rows using all three columns ('BOOK', 'PAGE', 'PARCEL'): {n_dupes_all}")

n_dupes_book_page = parcel_gdf.duplicated(subset=["BOOK", "PAGE"]).sum()
print(f"Number of duplicate rows using just 'BOOK' and 'PAGE': {n_dupes_book_page}")

if "GRPPCLID" in parcel_gdf.columns:
    n_dupes_grppclid = parcel_gdf.duplicated(subset=["GRPPCLID"]).sum()
    print(f"Number of duplicate rows using 'GRPPCLID': {n_dupes_grppclid}")
else:
    print("No 'GRPPCLID' column found in parcel_gdf.")

In [ ]:
for col in ["BOOK", "PAGE", "PARCEL"]:
    n_dupes = parcel_gdf.duplicated(subset=[col]).sum()
    print(f"Number of duplicate rows in '{col}': {n_dupes}")

# Create a column that concatenates MAPNUM, BLKNUM, and PARCELNUM as a string (with underscore as separator)
parcel_gdf["NUMS_CONCAT"] = (
    parcel_gdf["BOOK"].astype(str) + "_" +
    parcel_gdf["PAGE"].astype(str) + "_" +
    parcel_gdf["PARCEL"].astype(str)
)

# Identify duplicate rows based on NUMS_CONCAT
dupe_mask = parcel_gdf.duplicated(subset=["NUMS_CONCAT"], keep=False)
num_dupes = dupe_mask.sum()
print(f"Number of duplicate rows by NUMS_CONCAT: {num_dupes}")

# Print value counts of D_CLASS_CN among those duplicates
print("Value counts of D_CLASS_CN among duplicated NUMS_CONCAT rows:")
print(parcel_gdf.loc[dupe_mask, "CLASS"].value_counts(dropna=False))


In [ ]:

import osmnx as ox
import geopandas as gpd
from shapely.geometry import shape

# Get the polygon boundary for Cincinnati, OH
cincinnati_boundary = ox.geocode_to_gdf("Cincinnati, Ohio, USA")

print(cincinnati_boundary)

# Make sure parcel_gdf is a GeoDataFrame with a known CRS
if not isinstance(parcel_gdf, gpd.GeoDataFrame):
    # Try to convert, assuming geometry column exists and is shapely or WKB
    try:
        parcel_gdf = gpd.GeoDataFrame(parcel_gdf, geometry="geometry")
    except Exception as e:
        # Fallback: try to force-generate a geometry column if needed
        raise RuntimeError(f"Unable to convert parcel_gdf to GeoDataFrame: {e}")

if parcel_gdf.crs is None:
    # EPSG:4326 is a reasonable default for parcels parsed from ESRI or WGS84
    parcel_gdf = parcel_gdf.set_crs("EPSG:4326", allow_override=True)

if cincinnati_boundary.crs != parcel_gdf.crs:
    cincinnati_boundary = cincinnati_boundary.to_crs(parcel_gdf.crs)

# Restrict parcel_gdf to those geometries inside cincinnati_boundary (use first polygon)
boundary_geom = cincinnati_boundary.geometry.iloc[0]

if len(parcel_gdf) > 0:
    # Validate geometry: check for nulls and validity
    geometry_valid_mask = parcel_gdf["geometry"].notnull() & parcel_gdf["geometry"].apply(
        lambda x: getattr(x, "is_valid", False)
    )
    inside_mask = geometry_valid_mask & parcel_gdf["geometry"].apply(
        lambda geom: geom.within(boundary_geom) if geom is not None else False
    )
    n_total = len(parcel_gdf)
    n_inside = inside_mask.sum()
    percent_inside = 100 * n_inside / n_total if n_total > 0 else 0
    print(f"{n_inside} rows ({percent_inside:.2f}% of total) are within the Cincinnati boundary.")
    parcel_gdf = parcel_gdf[inside_mask].copy()
else:
    print("parcel_gdf is empty after CRS check.")



In [ ]:
# Print full value counts without truncation
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(parcel_gdf["CLASS"].value_counts(dropna=False))


In [ ]:
# Collapse duplicates on GRPPCLID by summing all all-caps columns (numerics),
# taking the first string/categorical from other columns, and unioning geometries.
from shapely.ops import unary_union
from shapely.geometry import MultiPolygon
import numpy as np

if "GRPPCLID" in parcel_gdf.columns:
    subset_cols = ["GRPPCLID"]

    # Identify all-caps columns for summing
    all_caps_cols = [
        col for col in parcel_gdf.columns
        if col.isupper() and parcel_gdf[col].dtype.kind in "biufcO"
    ]
    # Only sum if dtype is numeric
    numeric_sum_cols = [
        col for col in all_caps_cols if np.issubdtype(parcel_gdf[col].dtype, np.number)
    ]
    categorical_cols = [
        col for col in parcel_gdf.columns
        if col != "geometry" and col not in subset_cols and col not in numeric_sum_cols
    ]

    print("Numeric columns being summed:", numeric_sum_cols)
    print("Categorical columns being treated as 'first':", categorical_cols)

    # Handle geometry union
    def collapse_geoms(geoms):
        geoms = [g for g in geoms if g is not None]
        if not geoms:
            return None
        out = []
        while geoms:
            ref = geoms.pop(0)
            group = [ref]
            rest = []
            for g in geoms:
                if ref.intersects(g) or ref.touches(g) or ref.equals(g):
                    group.append(g)
                else:
                    rest.append(g)
            unioned = unary_union(group)
            out.append(unioned)
            geoms = rest
        if len(out) == 1:
            return out[0]
        # flatten out to MultiPolygon where possible
        polygons = []
        for g in out:
            if g.geom_type == "Polygon":
                polygons.append(g)
            elif g.geom_type == "MultiPolygon":
                polygons.extend(g.geoms)
            else:
                polygons.append(g)
        return MultiPolygon(polygons)

    # Build aggregation dictionary
    agg_dict = {}
    for col in parcel_gdf.columns:
        if col in subset_cols:
            continue
        elif col in numeric_sum_cols:
            agg_dict[col] = "sum"
        elif col == "geometry":
            agg_dict["geometry"] = collapse_geoms
        else:
            agg_dict[col] = "first"

    parcel_gdf_collapsed = parcel_gdf.groupby(subset_cols, dropna=False).agg(agg_dict).reset_index()
    print(f"Collapsed dataframe now has {len(parcel_gdf_collapsed)} rows (from {len(parcel_gdf)} original rows).")

    # Make sure GeoDataFrame type & CRS preserved
    from geopandas import GeoDataFrame
    was_geodf = isinstance(parcel_gdf, GeoDataFrame)
    crs = getattr(parcel_gdf, "crs", None) if was_geodf else None
    if "geometry" in parcel_gdf_collapsed.columns:
        parcel_gdf_collapsed = GeoDataFrame(parcel_gdf_collapsed, geometry="geometry", crs=crs)
    parcel_gdf = parcel_gdf_collapsed.copy()

    # Print how many duplicate sets were collapsed
    value_counts = parcel_gdf.groupby(subset_cols).size()
    n_multi = (value_counts > 1).sum()
    print(f"Collapsed {n_multi} sets of duplicate {subset_cols}.")

else:
    print(f"'GRPPCLID' column not found in parcel_gdf.")


In [ ]:
def categorize_property_type_from_class(class_value):
    """
    Categorizes property based on Hamilton County/Cincinnati's CLASS 3-digit code.
    """
    # Allow for string or int input
    try:
        class_str = str(class_value).strip().upper()
        if class_str.endswith("D"):  # Allow for things like 561 D
            class_code = class_str
        else:
            class_code = str(int(float(class_value))).zfill(3)
    except Exception:
        class_code = str(class_value).strip().upper()

    # Big-picture category buckets using CLASS field
    residential = {
        '500', '501', '502', '503', '504', '505', '507', '508', '510', '517', '520', '530', '550', '551',
        '552', '553', '554', '555', '556', '558', '580', '561D', '569', '599'
    }
    single_family = {'510'}
    two_family = {'517'}
    three_family = {'520'}
    condo = {'530', '550', '551', '552', '553', '554', '555', '556', '558'}
    manufactured_home = {'580', '561D'}
    res_low_income = {'569'}
    res_vacant = {'500', '501', '502', '503', '504', '505', '507'}  # vacant, forestry, street
    res_other = {'599'}

    commercial = {
        '400', '401', '402', '403', '404', '405', '406', '407', '410', '411', '412', '413', '415', '416',
        '417', '418', '419', '420', '421', '422', '424', '425', '427', '428', '429', '430', '431', '432',
        '433', '434', '435', '436', '439', '440', '441', '442', '444', '445', '447', '448', '449', '450',
        '452', '453', '454', '455', '456', '460', '461', '462', '463', '464', '465', '469', '470', '471',
        '480', '482', '488', '489', '490', '495', '498', '499'
    }
    comm_apartment_4_19 = {'401'}
    comm_apartment_20_39 = {'402'}
    comm_apartment_40_up = {'403'}
    comm_office = {'447', '448', '449', '431', '432', '445'}
    comm_retail = {'420', '421', '422', '424', '425', '427', '428', '429', '404', '405', '406', '430', '433', '434'}
    comm_industrial = {'480', '482', '488', '489'}
    comm_parking_garage = {'455', '456'}
    comm_low_income = {'469'}
    comm_vacant = {'400'}
    comm_other = {'499'}

    industrial = {
        '300', '307', '310', '317', '320', '330', '340', '350', '351', '352', '360', '370', '380', '389',
        '390', '399'
    }
    industrial_vacant = {'300'}
    warehousing = {'350', '351', '352'}
    manufacturing = {'310', '317', '320', '330', '340', '370'}
    industrial_other = {'380', '389', '390', '399'}

    ag = {
        '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', 
        '114', '115', '116', '117', '120', '121', '122', '123', '124', '190', '199'
    }
    ag_other = {'190', '199'}

    extraction = {
        '210', '220', '230', '240', '250', '260'
    }
    extraction_other = {'250', '260'}

    publicly_owned = {
        '600', '610', '620', '630', '640', '645', '650', '660', '670', '680', '685', '690'
    }

    abated = {
        '700', '710', '720', '730', '740', '750', '760'
    }
    public_util = {
        '800', '810', '820', '830', '840', '850', '860', '870', '880', '881'
    }

    # Stepwise assignment
    if class_code in single_family:
        return "Single Family"
    elif class_code in two_family:
        return "Two Family"
    elif class_code in three_family:
        return "Three Family"
    elif class_code in condo:
        return "Condo/PUD"
    elif class_code in manufactured_home:
        return "Manufactured Home"
    elif class_code in res_low_income:
        return "Residential Low-Income Tax Credit"
    elif class_code in res_vacant:
        return "Residential Vacant"
    elif class_code in residential or (class_code.endswith('D') and class_code.startswith('561')):
        return "Other Residential"
    elif class_code in comm_apartment_4_19:
        return "Multifamily (4-19 units)"
    elif class_code in comm_apartment_20_39:
        return "Multifamily (20-39 units)"
    elif class_code in comm_apartment_40_up:
        return "Multifamily (40+ units)"
    elif class_code in comm_low_income:
        return "Commercial Low-Income Tax Credit"
    elif class_code in comm_office:
        return "Commercial Office"
    elif class_code in comm_retail:
        return "Commercial Retail"
    elif class_code in comm_parking_garage:
        return "Parking Garage"
    elif class_code in comm_vacant:
        return "Commercial Vacant"
    elif class_code in comm_other:
        return "Other Commercial"
    elif class_code in commercial:
        return "Commercial"
    elif class_code in warehousing:
        return "Industrial Warehouse"
    elif class_code in manufacturing:
        return "Manufacturing"
    elif class_code in industrial_vacant:
        return "Industrial Vacant"
    elif class_code in industrial_other:
        return "Other Industrial"
    elif class_code in industrial:
        return "Industrial"
    elif class_code in ag_other:
        return "Other Agriculture"
    elif class_code in ag:
        return "Agriculture"
    elif class_code in extraction_other:
        return "Other Extraction"
    elif class_code in extraction:
        return "Extraction"
    elif class_code in publicly_owned:
        return "Publicly Owned"
    elif class_code in abated:
        return "Abated"
    elif class_code in public_util:
        return "Public Utility"
    else:
        # Try to catch plausible class code ranges by type as string fallback
        try:
            val = int("".join(c for c in class_code if c.isdigit()))
            if 100 <= val <= 199:
                return "Agriculture"
            elif 200 <= val <= 299:
                return "Extraction"
            elif 300 <= val <= 399:
                return "Industrial"
            elif 400 <= val <= 499:
                return "Commercial"
            elif 500 <= val <= 599:
                return "Residential"
            elif 600 <= val <= 699:
                return "Publicly Owned"
            elif 700 <= val <= 799:
                return "Abated"
            elif 800 <= val <= 899:
                return "Public Utility"
        except Exception:
            pass

    return "Other"

# Assign to parcel_gdf['PROPERTY_CATEGORY'] using the new function and the CLASS column
parcel_gdf['PROPERTY_CATEGORY'] = parcel_gdf['CLASS'].apply(categorize_property_type_from_class)

In [ ]:
# Exclude all rows where PROPERTY_CATEGORY is "Publicly Owned" (case-insensitive)
parcel_gdf = parcel_gdf[~(parcel_gdf["PROPERTY_CATEGORY"].str.lower() == "publicly owned")]


In [ ]:
# -----------------------------
# 1) Clone dataframe
# -----------------------------
export_gdf = parcel_gdf.copy()

# -----------------------------
# 2) Exclude fully exempt parcels and flag
# -----------------------------



# -----------------------------
# 3) Use appraised land/improvement values (fallback to assessed)
# -----------------------------
if "MKTLND" in export_gdf.columns:
    export_gdf["land_value"] = export_gdf["MKTLND"]
else:
    raise ValueError("Need land value field for Denver parcels.")

if "MKTIMP" in export_gdf.columns:
    export_gdf["improvement_value"] = export_gdf["MKTIMP"]
else:
    export_gdf["improvement_value"] = 0

# -----------------------------
# 4) Ensure PROPERTY_CATEGORY exists
# -----------------------------
if "PROPERTY_CATEGORY" not in export_gdf.columns:
    if "PROP_CLASS" in export_gdf.columns:
        export_gdf["PROPERTY_CATEGORY"] = np.where(
            export_gdf["PROP_CLASS"].astype(str).str.contains("VAC", case=False, na=False),
            "Vacant Land",
            "Other"
        )
    else:
        export_gdf["PROPERTY_CATEGORY"] = "Other"

export_gdf["property_land_use_category"] = export_gdf["PROPERTY_CATEGORY"]

# -----------------------------
# 5) Refined land use classification
# -----------------------------
def categorize_property_refined(row):
    cat = str(row["PROPERTY_CATEGORY"])
    if "Vacant" in cat:
        return "Vacant"
    elif "Parking" in cat:
        return "Parking Lot"
    elif row["improvement_value"] < 0.5 * (row["land_value"] + row["improvement_value"]):
        return "Underdeveloped"
    else:
        return None

export_gdf["property_land_use_refined"] = export_gdf.apply(categorize_property_refined, axis=1)

# -----------------------------
# 6) Compute parcel area sqft
# -----------------------------
import numpy as np
from pyproj import Geod

geod = Geod(ellps="WGS84")

def geodesic_area_sqft(geom):
    if geom is None or geom.is_empty:
        return np.nan

    gtype = geom.geom_type
    if gtype == "Polygon":
        lon, lat = geom.exterior.coords.xy
        area_m2, _ = geod.polygon_area_perimeter(lon, lat)
        return abs(area_m2) * 10.763910416709722  # m² -> ft²

    if gtype == "MultiPolygon":
        return sum(geodesic_area_sqft(p) for p in geom.geoms)

    return np.nan  # points/lines etc.

# (optional but helps with occasional self-intersections)
export_gdf["geometry"] = export_gdf["geometry"].apply(
    lambda g: g if g is None or g.is_valid else g.buffer(0)
)

export_gdf["area_sqft"] = export_gdf["geometry"].apply(geodesic_area_sqft)

# guardrail to prevent insane $/sqft from slivers
export_gdf.loc[export_gdf["area_sqft"] < 1, "area_sqft"] = np.nan

# -----------------------------
# 7) Per sqft metrics and full market value per sqft
# -----------------------------
if "MARKET_TOTAL_VALUE" in export_gdf.columns:
    export_gdf["full_market_value"] = export_gdf["MARKET_TOTAL_VALUE"]
else:
    export_gdf["full_market_value"] = export_gdf.get("land_value", 0) + export_gdf.get("improvement_value", 0)

export_gdf["full_market_value_per_sqft"] = export_gdf["full_market_value"] / export_gdf["area_sqft"]
export_gdf["land_value_per_sqft"] = export_gdf["land_value"] / export_gdf["area_sqft"]
export_gdf["improvement_value_per_sqft"] = export_gdf["improvement_value"] / export_gdf["area_sqft"]

# -----------------------------
# 8) Derived improvement/land ratios
# -----------------------------
export_gdf = add_improvement_ratio_fields(
    export_gdf,
    land_col="land_value",
    improvement_col="improvement_value"
)

# -----------------------------
# Save the parcel link, ensuring it's present in the export data
# -----------------------------
if "parcel_link" not in export_gdf.columns:
    export_gdf["parcel_link"] = np.nan

# Maintain legacy 'link' field for UI compatibility
if "link" not in export_gdf.columns:
    export_gdf["link"] = export_gdf["parcel_link"]

# -----------------------------
# 9) Select columns (no current_tax), now including parcel_link and link
# -----------------------------
columns_to_export = [
    "geometry",
    "exemption_flag",
    "property_land_use_category",
    "property_land_use_refined",
    "full_market_value",
    "full_market_value_per_sqft",
    "land_value",
    "land_value_per_sqft",
    "improvement_value",
    "improvement_value_per_sqft",
    "TLLDIMPROV",
    "IMPR_LAND_RATIO",
    "IMPR_LAND_PCT",
    "IMPR_PCT_TOTAL",
    "parcel_link",
    "link"
]

# Guarantee the export schema even if some columns are absent
for col in columns_to_export:
    if col not in export_gdf.columns:
        export_gdf[col] = np.nan

export_final = export_gdf[columns_to_export].rename(columns={
    "land_value": "current_full_land_value"
})

# Ensure geometry validity
export_final["geometry"] = export_final["geometry"].apply(
    lambda geom: geom if geom is None or geom.is_valid else geom.buffer(0)
)

# Ensure CRS is EPSG:4326
export_final = gpd.GeoDataFrame(export_final, geometry="geometry", crs=export_gdf.crs)
if export_final.crs is None or export_final.crs.to_epsg() != 4326:
    export_final = export_final.to_crs("EPSG:4326")
    print("✅ Converted to EPSG:4326")

# -----------------------------
# 10) Save Parquet: both canonical and dated version
# -----------------------------
canonical_path = os.path.join(DATA_DIR, "cincinnati-oh-parcels.parquet")
today_str = datetime.now().strftime("%Y_%m_%d")
dated_path = os.path.join(DATA_DIR, f"cincinnati-oh-parcels_{today_str}.parquet")

export_final.to_parquet(canonical_path, index=False)
export_final.to_parquet(dated_path, index=False)

print(f"✅ Saved export parquet: {canonical_path}")
print(f"✅ Also saved dated version: {dated_path}")
print("Export columns:", export_final.columns.tolist())
print("\nRefined category counts:")
print(export_final["property_land_use_refined"].value_counts(dropna=False))


In [ ]:
print("CRS:", export_gdf.crs)
print("Bounds:", export_gdf.total_bounds)  # [minx, miny, maxx, maxy]
print(export_gdf.geometry.geom_type.value_counts())


In [ ]:
# Optional: upload export_final to dev Azure blob
upload_dev = True

if upload_dev:
    from azure.storage.blob import BlobServiceClient

    connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    if not connection_string:
        raise ValueError(
            "Set AZURE_STORAGE_CONNECTION_STRING or update connection_string before upload."
        )

    container = os.getenv("AZURE_DEV_CONTAINER", "parquets-dev")
    blob_name = "cincinnati-oh-parcels.parquet"
    local_path = os.path.join(DATA_DIR, blob_name)

    if not os.path.exists(local_path):
        raise FileNotFoundError(f"Local parquet not found: {local_path}")

    blob_service = BlobServiceClient.from_connection_string(connection_string)
    container_client = blob_service.get_container_client(container)

    with open(local_path, "rb") as handle:
        container_client.upload_blob(name=blob_name, data=handle, overwrite=True)

    print(f"✅ Uploaded {local_path} -> {container}/{blob_name}")
else:
    print("upload_dev is False; skipping upload.")


In [ ]:
upload_dev_pmtiles = True  # Only run PMTiles conversion/upload if this is True

if upload_dev_pmtiles:
    # Convert parquet to PMTiles and upload to Azure
    import subprocess
    import sys
    from pathlib import Path

    # Get the path to the conversion script
    # Notebook is in data/jurisidictions/, script is in data/scripts/
    # Script expects to be run from PROJECT ROOT (where data/ is a subdirectory)
    notebook_dir = Path.cwd()  # Current working directory when notebook runs

    # Find project root by looking for data/scripts/ directory
    current = notebook_dir
    project_root = None
    while current.parent != current:
        if (current / "data" / "scripts" / "parquet_to_pmtiles.py").exists():
            project_root = current
            break
        current = current.parent

    if not project_root:
        # Fallback: assume project root is 2 levels up from jurisidictions
        project_root = notebook_dir.parent.parent if notebook_dir.name == "jurisidictions" else notebook_dir.parent

    script_path = project_root / "data" / "scripts" / "parquet_to_pmtiles.py"

    if not script_path.exists():
        raise FileNotFoundError(f"Could not find parquet_to_pmtiles.py script at: {script_path}")

    # Verify parquet file exists at expected location (relative to project root)
    expected_parquet = project_root / "data" / "jurisidictions" / "data" / "cincinnati" / "cincinnati-oh-parcels.parquet"
    print(f"Running PMTiles conversion script: {script_path}")
    print(f"Project root (working directory): {project_root}")
    print(f"Expected parquet: {expected_parquet}")
    print(f"Parquet exists: {expected_parquet.exists()}")

    # Run the conversion script
    cmd = [
        sys.executable,
        str(script_path),
        "--city", "cincinnati",
        "--upload",
        "--overwrite"
    ]

    # Check if connection string is set
    connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    if not connection_string:
        print("⚠️  AZURE_STORAGE_CONNECTION_STRING not set. Skipping upload.")
        cmd.remove("--upload")

    # Run from PROJECT ROOT (script looks for data/jurisidictions/data/cincinnati/cincinnati-oh-parcels.parquet relative to project root)
    result = subprocess.run(cmd, cwd=str(project_root), capture_output=True, text=True)

    if result.returncode == 0:
        print("✅ PMTiles conversion and upload completed successfully!")
        print("✅ PMTiles file: cincinnati-oh-parcels.pmtiles")
        print("✅ Metadata file: cincinnati-oh-parcels-metadata.json")
        if result.stdout:
            print("\nScript output:")
            print(result.stdout)
    else:
        print(f"❌ PMTiles conversion failed with exit code {result.returncode}")
        if result.stderr:
            print(f"\nError output:\n{result.stderr}")
        if result.stdout:
            print(f"\nStandard output:\n{result.stdout}")
else:
    print("upload_dev_pmtiles is False; skipping PMTiles conversion and upload.")
